In [ ]:
import glob
import os
import pandas as pd

OUT_PATH = "submission.csv"
CONST_PROB = 0.02

taxonomy_candidates = glob.glob("/kaggle/input/**/taxonomy.csv", recursive=True)
if not taxonomy_candidates:
    raise FileNotFoundError("taxonomy.csv not found under /kaggle/input")
TAXONOMY_PATH = sorted(taxonomy_candidates)[0]

taxonomy = pd.read_csv(TAXONOMY_PATH)
class_cols = taxonomy["primary_label"].tolist()

# Code competitions may not expose hidden test files during a plain run.
# Prefer sample_submission row_id as the robust source of required rows.
sample_candidates = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
if sample_candidates:
    SAMPLE_PATH = sorted(sample_candidates)[0]
    rows = pd.read_csv(SAMPLE_PATH)["row_id"].tolist()
else:
    soundscape_dirs = glob.glob("/kaggle/input/**/test_soundscapes", recursive=True)
    if not soundscape_dirs:
        raise FileNotFoundError("Neither sample_submission.csv nor test_soundscapes was found.")
    TEST_GLOB = os.path.join(sorted(soundscape_dirs)[0], "*.ogg")
    test_files = sorted(glob.glob(TEST_GLOB))
    if not test_files:
        raise FileNotFoundError(f"No test soundscape files matched {TEST_GLOB}")
    rows = []
    for path in test_files:
        stem = os.path.splitext(os.path.basename(path))[0]
        for end_sec in range(5, 61, 5):
            rows.append(f"{stem}_{end_sec}")

submission = pd.DataFrame({"row_id": rows, **{c: CONST_PROB for c in class_cols}})

submission.to_csv(OUT_PATH, index=False)
print(submission.shape)
print(f"Wrote {OUT_PATH}")
